#### Data loading and cleaning

In [3]:
import pandas as pd

In [11]:
print(df_team_information.loc[df_team_information['team'] == 'Iraq', 'squad_list'])

33    https://www.espn.co.uk/football/team/squad/_/i...
Name: squad_list, dtype: str


In [12]:
import pandas as pd

all_dfs = []

urls = df_team_information['squad_list'].unique()

for url in urls:
    try:
        print(f"Processing: {url}")
        
        tables = pd.read_html(url)

        # Combine the two tables on the page
        df = pd.concat(tables, ignore_index=True)

        # Optional: remove repeated header rows
        df = df[df[df.columns[0]] != df.columns[0]]
        df = df.dropna(how="all")

        # Add team identifier (from URL)
        team_id = url.split("/")[-1]
        df["team_id"] = team_id

        all_dfs.append(df)

    except Exception as e:
        print(f"Failed for {url}: {e}")

# Combine all teams into one DataFrame
final_df = pd.concat(all_dfs, ignore_index=True)

# Save to CSV
final_df.to_csv("data/all_world_cup_squads.csv", index=False)

print("Done! File saved as all_world_cup_squads.csv")

Processing: https://www.espn.co.uk/football/team/squad/_/id/450
Processing: https://www.espn.co.uk/football/team/squad/_/id/203
Processing: https://www.espn.co.uk/football/team/squad/_/id/467
Processing: https://www.espn.co.uk/football/team/squad/_/id/451
Processing: https://www.espn.co.uk/football/team/squad/_/id/452
Processing: https://www.espn.co.uk/football/team/squad/_/id/206
Processing: https://www.espn.co.uk/football/team/squad/_/id/4398
Processing: https://www.espn.co.uk/football/team/squad/_/id/475
Processing: https://www.espn.co.uk/football/team/squad/_/id/205
Processing: https://www.espn.co.uk/football/team/squad/_/id/2654
Processing: https://www.espn.co.uk/football/team/squad/_/id/2869
Processing: https://www.espn.co.uk/football/team/squad/_/id/580
Processing: https://www.espn.co.uk/football/team/squad/_/id/628
Processing: https://www.espn.co.uk/football/team/squad/_/id/210
Processing: https://www.espn.co.uk/football/team/squad/_/id/465
Processing: https://www.espn.co.uk/fo

,Name,team,POS,Age,HT,WT,NAT,APP,SUB,SV,...,Unnamed: 15,G,SH,ST,team_id,P,SB,S,GC,SO
0,Martin Jedlicka,Czechia,G,28,1.88 m,--,Czechia,0,0,0.0,...,NaN,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN
1,Matej Kovar,Czechia,G,25,1.96 m,87 kg,Czechia,9,0,18.0,...,NaN,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN
2,Lukas Hornicek,Czechia,G,23,1.93 m,88 kg,Czechia,0,0,0.0,...,NaN,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN
3,Ladislav Krejcí,Czechia,D,27,1.91 m,87 kg,Czechia,9,0,NaN,...,NaN,2,8,5,450,NaN,NaN,NaN,NaN,NaN
4,Vladimír Coufal,Czechia,D,33,1.75 m,76 kg,Czechia,9,0,NaN,...,NaN,1,3,3,450,NaN,NaN,NaN,NaN,NaN


In [13]:
df_combined = pd.read_csv('data/all_world_cup_squads.csv')
df_combined['team_id'] = df_combined['team_id'].astype('int64')

df_combined = df_combined.merge(
    df_team_information[["team_id", "team"]],
    on="team_id",
    how="left"
)

# reorder columns to put team in second place
cols = list(df_combined.columns)
cols.remove('team')
cols.insert(1, 'team')
df_combined = df_combined[cols]

## clean column names
df_combined.columns = (
    df_combined.columns
    .str.strip()              # remove leading/trailing whitespace
    .str.lower()              # convert to lowercase
    .str.replace(r'\s+', '_', regex=True)  # replace spaces with underscores
)

# add new column with player name and team, to be used in search
df_combined['player_name_team'] = df_combined['name'] + " " + df_combined['team'] + " headshot"

# delete unnamed empty column
df_combined = df_combined.drop('unnamed:_15', axis=1)

df_combined.head()




,name,team,pos,age,ht,wt,nat,app,sub,sv,...,g,sh,st,team_id,p,sb,s,gc,so,player_name_team
0,Martin Jedlicka,Czechia,G,28,1.88 m,--,Czechia,0,0,0.0,...,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN,Martin Jedlicka Czechia headshot
1,Matej Kovar,Czechia,G,25,1.96 m,87 kg,Czechia,9,0,18.0,...,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN,Matej Kovar Czechia headshot
2,Lukas Hornicek,Czechia,G,23,1.93 m,88 kg,Czechia,0,0,0.0,...,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN,Lukas Hornicek Czechia headshot
3,Ladislav Krejcí,Czechia,D,27,1.91 m,87 kg,Czechia,9,0,NaN,...,2,8,5,450,NaN,NaN,NaN,NaN,NaN,Ladislav Krejcí Czechia headshot
4,Vladimír Coufal,Czechia,D,33,1.75 m,76 kg,Czechia,9,0,NaN,...,1,3,3,450,NaN,NaN,NaN,NaN,NaN,Vladimír Coufal Czechia headshot


In [20]:
df_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 1277 entries, 0 to 1276
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   name              1277 non-null   str  
 1   team              1277 non-null   str  
 2   pos               1277 non-null   str  
 3   age               1277 non-null   str  
 4   ht                1277 non-null   str  
 5   wt                1277 non-null   str  
 6   nat               1277 non-null   str  
 7   app               166 non-null    str  
 8   sub               166 non-null    str  
 9   sv                21 non-null     str  
 10  ga                21 non-null     str  
 11  a                 1277 non-null   str  
 12  fc                1277 non-null   str  
 13  fa                1277 non-null   str  
 14  yc                1277 non-null   str  
 15  rc                1277 non-null   str  
 16  g                 1120 non-null   str  
 17  sh                1120 non-null   str  
 18 

#### Websites for downloading pictures:

* https://fbref.com/en/ has one picture per player
* https://en.soccerwiki.org one picture per player

In [21]:
teams_downloaded = pd.DataFrame(df_combined['team_id'].unique().astype('int64'), columns=['team_id'])
teams_included = pd.DataFrame(df_team_information['team_id'].unique().astype('int64'), columns=['team_id'])

# Get all unique team_ids from both DataFrames
all_teams = pd.DataFrame(
    set(teams_downloaded["team_id"]).union(set(teams_included["team_id"])),
    columns=["team_id"]
)

# Add flags
all_teams["in_downloaded"] = all_teams["team_id"].isin(teams_downloaded["team_id"])
all_teams["in_included"] = all_teams["team_id"].isin(teams_included["team_id"])

for team_id in all_teams.loc[
    (~all_teams["in_downloaded"]) | (~all_teams["in_included"]),
    "team_id"
]:
    print(f"The team ID value {team_id} is not included in both lists")

The team ID value 11678 is not included in both lists


In [22]:
df_combined.head()

,name,team,pos,age,ht,wt,nat,app,sub,sv,...,g,sh,st,team_id,p,sb,s,gc,so,player_name_team
0,Martin Jedlicka,Czechia,G,28,1.88 m,--,Czechia,0,0,0.0,...,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN,Martin Jedlicka Czechia headshot
1,Matej Kovar,Czechia,G,25,1.96 m,87 kg,Czechia,9,0,18.0,...,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN,Matej Kovar Czechia headshot
2,Lukas Hornicek,Czechia,G,23,1.93 m,88 kg,Czechia,0,0,0.0,...,NaN,NaN,NaN,450,NaN,NaN,NaN,NaN,NaN,Lukas Hornicek Czechia headshot
3,Ladislav Krejcí,Czechia,D,27,1.91 m,87 kg,Czechia,9,0,NaN,...,2,8,5,450,NaN,NaN,NaN,NaN,NaN,Ladislav Krejcí Czechia headshot
4,Vladimír Coufal,Czechia,D,33,1.75 m,76 kg,Czechia,9,0,NaN,...,1,3,3,450,NaN,NaN,NaN,NaN,NaN,Vladimír Coufal Czechia headshot


In [23]:
pd.DataFrame(df_combined['player_name_team'][df_combined['team'] == 'Panama'])

,player_name_team
1254,Luis Mejía Panama headshot
1255,Orlando Mosquera Panama headshot
1256,César Samudio Panama headshot
1257,Roderick Miller Panama headshot
1258,Éric Davis Panama headshot
1259,Andrés Andrade Panama headshot
1260,Michael Murillo Panama headshot
1261,César Blackman Panama headshot
1262,Jorge Gutiérrez Panama headshot
1263,José Córdoba Panama headshot
